In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches
import random
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import MinMaxScaler

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers


# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [3]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/marcor/Desktop/projects/InstaNexus


In [4]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

figures


In [22]:
# Path to the new raw data you want to test

INPUT_CSV = "inputs/bsa.csv"
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

RUN_NAME = Path(INPUT_CSV).stem

# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
CHAIN = "heavy"
REFERENCE_MODE = True
KMER_SIZE = 6
MIN_OVERLAP = 2
SIZE_THRESHOLD = 10
CONFIDENCE_THRESHOLD = 0.8
MIN_LENGTH = 7
MAX_LENGTH = 20
FDR_THRESHOLD = 0.5
MIN_IDENTITY = 0.8
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [6]:
#base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
#experiment_folder = base_output_folder / run_folder_name

run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")

2025-12-22 18:45:35,327 [INFO] Pipeline starting for run: [bsa @ dbg_weighted_c0.8_ks6_mo2_ts0_mi0.8_mm0]


In [7]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [8]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [9]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

Sample uses proteases: ['Chymotrypsin', 'Legumain', 'Krakatoa', 'Elastase', 'Trypsin', 'Papain', 'Thermo', 'ProtK', 'GluC', 'LysC']
Protein sequence length: 607 amino acids
Normalized protein sequence: MKWVTFLSLLLLFSSAYSRGVFRRDTHKSELAHRFKDLGEEHFKGLVLLAFSQYLQQCPFDEHVKLVNELTEFAKTCVADESHAGCEKSLHTLFGDELCKVASLRETYGDMADCCEKQEPERNECFLSHKDDSPDLPKLKPDPNTLCDEFKADEKKFWGKYLYELARRHPYFYAPELLYYANKYNGVFQECCQAEDKGACLLPKLETMREKVLASSARQRLRCASLQKFGERALKAWSVARLSQKFPKAEFVEVTKLVTDLTKVHKECCHGDLLECADDRADLAKYLCDNQDTLSSKLKECCDKPLLEKSHCLAEVEKDALPENLPPLTADFAEDKDVCKNYQEAKDAFLGSFLYEYSRRHPEYAVSVLLRLAKEYEATLEECCAKDDPHACYSTVFDKLKHLVDEPQNLLKQNCDQFEKLGEYGFQNALLVRYTRKVPQVSTPTLVEVSRSLGKVGTRCCTKPESERMPCTEDYLSLLLNRLCVLHEKTPVSEKVTKCCTESLVNRRPCFSALTPDETYVPKAFDEKLFTFHADLCTLPDTEKQLKKQTALVELLKHKPKATEEQLKTVMENFVAFVDKCCAADDKEACFAVEGPKLVVSTQTALA


In [10]:
original_data = pd.read_csv(INPUT_CSV)

cols_to_keep = [
    'experiment_name',
    'prediction_untokenised',
    'instanovo_token_log_probabilities',
    'calibrated_confidence',    
    'psm_q_value',
    'delta_mass_ppm',
    'Mass Error',               
    'is_missing_prosit_features', 
    'ion_match_intensity',
    'ion_matches',
    'iRT',
    'iRT error',
    'is_missing_irt_error',
    'predicted iRT',
    'margin',
    'entropy',
    'z-score'
    ]

data = original_data[cols_to_keep].copy()

data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = data.pop("protease")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

data = data.dropna(subset=["prediction_untokenised"])

data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

cleaned_preds_col = data.pop("cleaned_preds")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

cleaned_psms = data["cleaned_preds"].tolist()

filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

data = data[data["cleaned_preds"].isin(filtered_psms)]

data.drop(columns=['prediction_untokenised'], inplace=True)

data["mapped"] = data["cleaned_preds"].apply(
    lambda x: "True" if x in protein_norm else "False"
)

data = data[data['cleaned_preds'].str.len() >= MIN_LENGTH]

data = data[data['cleaned_preds'].str.len() <= MAX_LENGTH]

In [11]:
from scipy.stats import gaussian_kde

In [14]:
def plot_ridgeline_log_kde(
    df,
    protease_column="protease",
    conf_column="conf",
    protease_list=None,
    vertical_gap=0.5,
    width_ratio=1,
    figsize=None,
    cmap="viridis",
    custom_colors=None,
    title="",
    save_svg_path=FIGURES_DIR / f"{RUN_NAME}_ridgeline_plot.svg",
    x_limits=(0, 1)
):
    
    # 1. Applica lo stile globale
    visualization.set_publication_style()

    # 2. Gestione intelligente delle dimensioni
    # Se l'utente non fornisce figsize esplicito, usiamo l'helper standard
    if figsize is None:
        figsize = visualization.get_figsize(width_ratio=width_ratio)

    # 3. Preparazione Dati
    if protease_list is None:
        protease_list = sorted(df[protease_column].dropna().unique())
    
    x_vals = np.linspace(x_limits[0], x_limits[1], 500)

    # Gestione Colori
    if custom_colors is not None:
        colors = [custom_colors.get(p, "#46473E") for p in protease_list]
    else:
        cmap_obj = plt.get_cmap(cmap)
        colors = cmap_obj(np.linspace(0, 1, len(protease_list)))

    # Calcolo KDE
    densities = []
    valid_proteases = []
    
    for p in protease_list:
        subset = df[df[protease_column] == p][conf_column].dropna()
        if len(subset) < 2: continue
            
        kde = gaussian_kde(subset)
        y = kde(x_vals)
        y_log = np.log10(y + 1e-6)
        densities.append(y_log)
        valid_proteases.append(p)

    if not densities:
        print("Dati insufficienti per il plot.")
        return

    global_min = np.min([d.min() for d in densities])

    fig, ax = plt.subplots(figsize=figsize)

    for i, (protease, y_log) in enumerate(zip(valid_proteases, densities)):
        y_shifted = y_log - global_min
        offset = i * vertical_gap
        final_curve = y_shifted + offset
        baseline = offset 
        
        ax.fill_between(
            x_vals, baseline, final_curve, 
            color=colors[i], alpha=0.9, zorder=len(valid_proteases)-i
        )
        ax.plot(
            x_vals, final_curve, 
            color="white", lw=1.2, zorder=len(valid_proteases)-i+0.1
        )
        
        ax.text(
            x_limits[1] + 0.02, offset + (y_shifted.max() * 0.2), 
            protease, va="center", ha="left", fontsize=12, color="#333333"
        )

    ax.set_xlabel("Confidence Score", fontsize=15)
    ax.set_xlim(x_limits)
    ax.set_yticks([])
    
    sns.despine(left=True, bottom=False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['bottom'].set_color("black")

    ax.set_title(title, pad=20, loc="left")

    if save_svg_path:
        os.makedirs(os.path.dirname(save_svg_path), exist_ok=True)
        
        plt.savefig(save_svg_path, format="svg", bbox_inches="tight", transparent=False)
        print(f"Plot saved: {save_svg_path}")

In [15]:
plot_ridgeline_log_kde(data, width_ratio=1)

Plot saved: figures/bsa_ridgeline_plot.svg


In [16]:
data = data[data["conf"] > CONFIDENCE_THRESHOLD]

data.reset_index(drop=True, inplace=True)

final_psms = data["cleaned_preds"].tolist()

In [17]:
final_psms[0:5] 

['VLHEKTPVSEK', 'LSHKDDSPDLPK', 'LSHKDDSPDLPK', 'HPTARHPEYA', 'LSHKDDSPDLPK']

## Protease optimization

In [18]:
ordered_proteases = data["protease"].value_counts().index.tolist()

In [20]:
ordered_proteases

['Chymotrypsin',
 'Elastase',
 'Thermo',
 'ProtK',
 'LysC',
 'Trypsin',
 'Legumain',
 'GluC',
 'Papain',
 'Krakatoa']

In [19]:
build_results = []

In [23]:
assembler = assembly.Assembler(
    mode="dbg_weighted",
    kmer_size=7,
    min_overlap=3,
    size_threshold=10,
    min_weight=2,
    refine_rounds=5
)

In [21]:
print(SIZE_THRESHOLD, MIN_IDENTITY)

0 0.8


In [24]:
for i in range(1, len(ordered_proteases) + 1):
    selected_proteases = ordered_proteases[:i]
    filtered_df = data[data["protease"].isin(selected_proteases)]

    sequences = filtered_df["cleaned_preds"].tolist()

    scaffolds = assembler.run(sequences=sequences, df_full=filtered_df)
    
    mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, SIZE_THRESHOLD, MIN_IDENTITY)
    df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_scaffolds)
    stat_scaffolds = helpers.compute_assembly_statistics(
        df=df_mapped,
        sequence_type=f"scaffolds",
        output_folder="outputs/_protease_opt_analysis",
        reference=protein_norm,
        fdr_threshold=FDR_THRESHOLD 
    )
    coverage_scaffolds = stat_scaffolds.get("coverage")
    build_results.append(
        {
            "n_proteases_used": i,
            "coverage_scaffolds": coverage_scaffolds,
        }
    )
    df_build = pd.DataFrame(build_results)

2025-12-22 18:53:57,750 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 117/117 [00:00<00:00, 159640.07it/s]
2025-12-22 18:53:57,769 [INFO] DBG produced 12 initial contigs.
2025-12-22 18:53:57,770 [INFO] Refining contigs using Overlap Graph (Bird's Eye View)...
2025-12-22 18:53:57,771 [INFO] DEBUG: Overlap Graph has 12 nodes and 1 edges.
2025-12-22 18:53:57,772 [INFO] Round 1: reduced to 11 scaffolds.
2025-12-22 18:53:57,772 [INFO] DEBUG: Overlap Graph has 11 nodes and 0 edges.
2025-12-22 18:53:57,773 [WARNING] No overlaps found between scaffolds! Check sliding logic.
2025-12-22 18:53:57,773 [INFO] Refinement converged at round 2.
2025-12-22 18:53:57,791 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 178/178 [00:00<00:00, 549768.86it/s]
2025-12-22 18:53:57,793 [INFO] DBG produced 16 initial contigs.
2025-12-22 18:53:57,793 [INFO] Refining contigs using Overlap Graph (Bird's Eye View)...
2

In [25]:
df_build

,n_proteases_used,coverage_scaffolds
0,1,0.256579
1,2,0.379934
2,3,0.389803
3,4,0.396382
4,5,0.434211
5,6,0.524671
6,7,0.570724
7,8,0.597039
8,9,0.598684
9,10,0.621711


In [26]:
JSON_DIR = Path("json")

In [31]:
def plot_pareto_coverage(
    df_build, 
    ordered_proteases, 
    protease_colors, 
    width_ratio=2,
    save_path=None
):

    visualization.set_publication_style()
    
    delta_values = df_build["coverage_scaffolds"].diff()
    delta_values.iloc[0] = df_build["coverage_scaffolds"].iloc[0]
    
    df_plot = pd.DataFrame({
        "protease": ordered_proteases, 
        "delta_scaffolds": delta_values.values
    })
    
    df_plot = df_plot.sort_values("delta_scaffolds", ascending=False)
    
    total_gain = df_plot["delta_scaffolds"].sum()
    df_plot["cum_coverage_pct"] = (
        (100 * df_plot["delta_scaffolds"].cumsum() / total_gain)
        if total_gain != 0 else 0.0
    )
    
    bar_colors = [protease_colors.get(p, "#999999") for p in df_plot["protease"]]

    figsize = visualization.get_figsize(width_ratio=width_ratio)
    _, ax1 = plt.subplots(figsize=figsize)

    ax1.bar(
        df_plot["protease"], 
        df_plot["delta_scaffolds"], 
        color=bar_colors,
        edgecolor="black",
        linewidth=1,
        zorder=2
    )
    
    ax1.set_ylabel("Δ coverage (scaffolds)", color="black")
    ax1.set_xlabel("Proteases")
    ax1.tick_params(axis='y', labelcolor="black")
    ax1.tick_params(axis='x', rotation=45)

    ax2 = ax1.twinx()
    
    ax2.plot(
        df_plot["protease"], 
        df_plot["cum_coverage_pct"], 
        color="black", 
        linestyle="--", 
        marker="o",        
        markersize=6,
        linewidth=2,
        label="Cumulative % gain",
        zorder=3
    )
    
    ax2.set_ylabel("Cumulative % coverage gain", color="black")
    ax2.set_ylim(0, 110) 
    
    ax1.grid(False)
    ax2.grid(False)

    for ax in [ax1, ax2]:
        ax.spines['top'].set_visible(False) 
        ax.spines['bottom'].set_color('black')
        ax.spines['left'].set_color('black')
        ax.spines['right'].set_color('black')
        ax.spines['bottom'].set_linewidth(1.5)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['right'].set_linewidth(1.5)

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        
        plt.savefig(save_path, format="svg", bbox_inches="tight")
        print(f"Plot: {save_path}")

In [28]:
with open(JSON_DIR / "protease_colors.json", "r") as f:
    protease_colors = json.load(f)

In [33]:

plot_pareto_coverage(df_build, ordered_proteases, protease_colors, width_ratio=2, save_path=f"{FIGURES_DIR}/subb_fig3c_{RUN_NAME}_pareto_scaffolds_{ASSEMBLY_MODE}.svg")

Plot: figures/subb_fig3c_bsa_pareto_scaffolds_dbg_weighted.svg


## Leave one out approach

In [ ]:
proteases

In [35]:
# proteases = final_df['protease'].unique()

build_results_l1o = []
for protease in proteases:
    # Exclude the current protease
    filtered_df = data[data["protease"] != protease]

    # Extract sequences from the filtered DataFrame (assuming the column "preds")
    sequences = filtered_df["cleaned_preds"].tolist()

    scaffolds = assembler.run(sequences=sequences, df_full=filtered_df)
    
    mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, SIZE_THRESHOLD, MIN_IDENTITY)
    df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_scaffolds)
    stat_scaffolds = helpers.compute_assembly_statistics(
        df=df_mapped,
        sequence_type=f"scaffolds",
        output_folder="outputs/_protease_opt_analysis",
        reference=protein_norm,
        fdr_threshold=FDR_THRESHOLD 
    )
    coverage_scaffolds = stat_scaffolds.get("coverage")
    build_results_l1o.append(
        {
            "excluded_protease": protease,
            "coverage_scaffolds": coverage_scaffolds,
        }
    )

2025-12-22 19:11:28,441 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 363/363 [00:00<00:00, 472103.05it/s]
2025-12-22 19:11:28,446 [INFO] DBG produced 32 initial contigs.
2025-12-22 19:11:28,446 [INFO] Refining contigs using Overlap Graph (Bird's Eye View)...
2025-12-22 19:11:28,450 [INFO] DEBUG: Overlap Graph has 32 nodes and 8 edges.
2025-12-22 19:11:28,451 [INFO] Round 1: reduced to 25 scaffolds.
2025-12-22 19:11:28,453 [INFO] DEBUG: Overlap Graph has 25 nodes and 0 edges.
2025-12-22 19:11:28,454 [WARNING] No overlaps found between scaffolds! Check sliding logic.
2025-12-22 19:11:28,454 [INFO] Refinement converged at round 2.
2025-12-22 19:11:28,484 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 341/341 [00:00<00:00, 439538.31it/s]
2025-12-22 19:11:28,488 [INFO] DBG produced 28 initial contigs.
2025-12-22 19:11:28,488 [INFO] Refining contigs using Overlap Graph (Bird's Eye View)...
2

In [41]:
df_build_2 = pd.DataFrame(build_results_l1o)


In [42]:
df_build_2

,excluded_protease,coverage_scaffolds
0,Chymotrypsin,0.601974
1,Legumain,0.587171
2,Krakatoa,0.598684
3,Elastase,0.595395
4,Trypsin,0.585526
5,Papain,0.620066
6,Thermo,0.613487
7,ProtK,0.601974
8,GluC,0.595395
9,LysC,0.597039


In [ ]:
def plot_leave_one_out_coverage(
    df, 
    protease_colors, 
    x_col="coverage_scaffolds", 
    y_col="excluded_protease",
    width_ratio=1,
    save_path=None
):

    visualization.set_publication_style()
    
    df_plot = df.sort_values(x_col, ascending=True).copy()
    
    bar_colors = [protease_colors.get(p, "#999999") for p in df_plot[y_col]]
    
    figsize = visualization.get_figsize(width_ratio=width_ratio)
    _, ax = plt.subplots(figsize=figsize)
    
    bars = ax.barh(
        df_plot[y_col], 
        df_plot[x_col], 
        color=bar_colors,
        edgecolor="black",
        linewidth=1,
        height=0.7
    )
    
    ax.set_xlabel("Scaffold coverage")
    ax.set_ylabel("Excluded protease")
    
    ax.set_xlim(0, 1)
    
    sns.despine(left=False, bottom=False, top=True, right=True)
    
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, format="svg", bbox_inches="tight")
        print(f"Plot salvato: {save_path}")

In [44]:
with open(JSON_DIR / "protease_colors.json", "r") as f:
     protease_colors = json.load(f)

output_file = Path(FIGURES_DIR) / f"subb_fig3d_{RUN_NAME}_coverage_leave_one_out.svg"
plot_leave_one_out_coverage(df_build_2, protease_colors, save_path=output_file)

Plot salvato: figures/subb_fig3d_bsa_coverage_leave_one_out.svg


In [47]:
def plot_leave_one_out_lollipop(
    df, 
    protease_colors, 
    x_col="coverage_scaffolds", 
    y_col="excluded_protease",
    width_ratio=1,
    save_path=None,
    x_range=None
):
    visualization.set_publication_style()
    
    df_plot = df.sort_values(x_col, ascending=True).copy()
    point_colors = [protease_colors.get(p, "#999999") for p in df_plot[y_col]]
    
    figsize = visualization.get_figsize(width_ratio=width_ratio)
    fig, ax = plt.subplots(figsize=figsize)
    
    x_start = x_range[0] if x_range else 0
    
    ax.hlines(
        y=df_plot[y_col],
        xmin=x_start,
        xmax=df_plot[x_col],
        color='grey',
        alpha=0.5,
        linewidth=1.5,
        zorder=1
    )
    
    ax.scatter(
        df_plot[x_col],
        df_plot[y_col],
        color=point_colors,
        s=100,
        edgecolor='black',
        linewidth=1,
        alpha=1,
        zorder=2
    )
    
    ax.set_xlabel("Scaffold coverage")
    ax.set_ylabel("Excluded protease")
    
    if x_range:
        ax.set_xlim(x_range)
    else:
        min_val = df_plot[x_col].min()
        max_val = df_plot[x_col].max()
        padding = (max_val - min_val) * 0.2
        ax.set_xlim(max(0, min_val - padding), max_val + padding/2)
        
    sns.despine(left=True, bottom=False, top=True, right=True)
    
    ax.grid(axis='x', linestyle='--', alpha=0.3, color='grey')
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(1.5)
    ax.tick_params(axis='y', length=0) 

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, format="svg", bbox_inches="tight")
        print(f"Plot salvato: {save_path}")

In [48]:
plot_leave_one_out_lollipop(df_build_2, protease_colors, save_path=f"{FIGURES_DIR}/subb_fig3d_leave_one_out_lollipop.svg", x_range=[0.5, 0.7])

Plot salvato: figures/subb_fig3d_leave_one_out_lollipop.svg


## Upset plot

In [49]:
upset_proteases = ["Legumain", "Chymotrypsin", "Trypsin", "Elastase"]

In [50]:
from itertools import combinations

all_combinations = []

# Generate all combinations of the proteases
for r in range(1, len(upset_proteases) + 1):
    all_combinations.extend(combinations(upset_proteases, r))

all_combinations = [list(comb) for comb in all_combinations]

print(f"All combinations of proteases: {all_combinations}")

All combinations of proteases: [['Legumain'], ['Chymotrypsin'], ['Trypsin'], ['Elastase'], ['Legumain', 'Chymotrypsin'], ['Legumain', 'Trypsin'], ['Legumain', 'Elastase'], ['Chymotrypsin', 'Trypsin'], ['Chymotrypsin', 'Elastase'], ['Trypsin', 'Elastase'], ['Legumain', 'Chymotrypsin', 'Trypsin'], ['Legumain', 'Chymotrypsin', 'Elastase'], ['Legumain', 'Trypsin', 'Elastase'], ['Chymotrypsin', 'Trypsin', 'Elastase'], ['Legumain', 'Chymotrypsin', 'Trypsin', 'Elastase']]


In [52]:
import math

def to_float_or_none(x):
    try:
        xf = float(x)
        if math.isnan(xf) or math.isinf(xf):
            return None
        return xf
    except (TypeError, ValueError):
        return None


def to_int_or_none(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        try:
            xf = float(x)
            if math.isnan(xf) or math.isinf(xf):
                return None
            return int(xf)
        except (TypeError, ValueError):
            return None

In [51]:
OUTPUT_DIR = Path("outputs/_protease_opt_analysis")

In [53]:
coverage_results = []
matrix_rows = []

for combo in all_combinations:
    specific_df = data[data["protease"].isin(combo)]
    sequences = specific_df["cleaned_preds"].tolist()

    scaffolds = assembler.run(sequences=sequences, df_full=filtered_df)
    
    mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, SIZE_THRESHOLD, MIN_IDENTITY)
    df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_scaffolds)
    stat_scaffolds = helpers.compute_assembly_statistics(
        df=df_mapped,
        sequence_type=f"scaffolds",
        output_folder="outputs/_protease_opt_analysis",
        reference=protein_norm,
        fdr_threshold=FDR_THRESHOLD 
    )


    reference_length = len(protein_norm)
    if len(df_mapped):
        df_mapped["sequence_length"] = (
            df_mapped["end"] - df_mapped["start"] + 1
        )
        covered_positions = set()
        for s, e in zip(df_mapped["start"], df_mapped["end"]):
            covered_positions.update(range(int(s) - 1, int(e)))
        coverage = (
            (len(covered_positions) / reference_length) if reference_length else 0.0
        )

        avg_len = to_float_or_none(df_mapped["sequence_length"].mean())
        min_len = to_int_or_none(df_mapped["sequence_length"].min())
        max_len = to_int_or_none(df_mapped["sequence_length"].max())
        total_seq = to_int_or_none(len(df_mapped))
    else:
        coverage, avg_len, min_len, max_len, total_seq = 0.0, None, None, None, 0

    statistics = {}
    statistics.update(
        {
            "reference_length": reference_length,
            "total_sequences": total_seq,
            "average_length": avg_len,
            "min_length": min_len,
            "max_length": max_len,
            "coverage": to_float_or_none(coverage),
        }
    )

    protease_str = "_".join(p.lower() for p in combo)
    file_name = f"scaffolds_{protease_str}_stats.json"
    output_path = OUTPUT_DIR / file_name

    with open(output_path, "w") as f:
        json.dump(statistics, f, indent=4)

    row = {
        prot: 1 if prot in combo else 0
        for prot in ["Legumain", "Chymotrypsin", "Elastase", "Trypsin"]
    }
    row["coverage"] = statistics["coverage"]
    matrix_rows.append(row)

presence_absence_df = pd.DataFrame(matrix_rows)

2025-12-22 20:29:53,315 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 58/58 [00:00<00:00, 333703.20it/s]
2025-12-22 20:29:53,319 [INFO] DBG produced 6 initial contigs.
2025-12-22 20:29:53,320 [INFO] Refining contigs using Overlap Graph (Bird's Eye View)...
2025-12-22 20:29:53,320 [INFO] DEBUG: Overlap Graph has 6 nodes and 0 edges.
2025-12-22 20:29:53,320 [WARNING] No overlaps found between scaffolds! Check sliding logic.
2025-12-22 20:29:53,321 [INFO] Refinement converged at round 1.
2025-12-22 20:29:53,336 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 117/117 [00:00<00:00, 568636.81it/s]
2025-12-22 20:29:53,338 [INFO] DBG produced 12 initial contigs.
2025-12-22 20:29:53,339 [INFO] Refining contigs using Overlap Graph (Bird's Eye View)...
2025-12-22 20:29:53,339 [INFO] DEBUG: Overlap Graph has 12 nodes and 1 edges.
2025-12-22 20:29:53,339 [INFO] Round 1: reduced to 11 scaffolds.
2025-

In [54]:
presence_absence_df

,Legumain,Chymotrypsin,Elastase,Trypsin,coverage
0,1,0,0,0,0.082372
1,0,1,0,0,0.257002
2,0,0,0,1,0.171334
3,0,0,1,0,0.217463
4,1,1,0,0,0.299835
5,1,0,0,1,0.247117
6,1,0,1,0,0.273476
7,0,1,0,1,0.369028
8,0,1,1,0,0.380560
9,0,0,1,1,0.344316


In [62]:
from upsetplot import UpSet
import matplotlib.pyplot as plt
from pathlib import Path

def plot_upset_coverage(
    df,
    ind_cols,
    bar_color,
    width_ratio=3,
    save_path=None
):
    visualization.set_publication_style()

    data_boolean = df.copy()
    for col in ind_cols:
        data_boolean[col] = data_boolean[col].astype(bool)

    indexed = data_boolean.set_index(ind_cols)

    figsize = visualization.get_figsize(width_ratio=width_ratio)
    fig = plt.figure(figsize=figsize)
    
    upset = UpSet(
        indexed,
        intersection_plot_elements=0,
        totals_plot_elements=0,
        subset_size="count",
        show_counts=False,
        show_percentages=False,
        element_size=50,
        orientation="horizontal",
    )

    upset.add_catplot(
        value="coverage",
        kind="bar",
        color=bar_color,
        width=0.5,
    )

    upset.plot(fig=fig)

    all_axes = fig.get_axes()
    cat_ax = all_axes[-1]

    cat_ax.grid(False)
    cat_ax.set_ylim(0, 1.0)
    
    cat_ax.spines["top"].set_visible(False)
    cat_ax.spines["right"].set_visible(False)
    cat_ax.spines["left"].set_visible(True)
    cat_ax.spines["bottom"].set_visible(False)
    
    cat_ax.spines["left"].set_color("black")
    cat_ax.spines["left"].set_linewidth(1.5)
    cat_ax.tick_params(axis="y", colors="black")

    for p in cat_ax.patches:
        cat_ax.annotate(
            f"{p.get_height():.2f}",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=10,
            color="black",
        )

    pos = cat_ax.get_position()
    cat_ax.set_position([pos.x0, pos.y0, pos.width, pos.height * 2.5])

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, format="svg", dpi=300, bbox_inches="tight")
        print(f"Plot saved: {save_path}")

In [57]:
with open(Path("json") / "colors.json", "r") as f:
    colors_dict = json.load(f)

run_color = colors_dict.get(RUN_NAME, )

In [58]:
run_color = colors_dict["bsa"]["scaffold"]

In [59]:
ind_cols = [col for col in presence_absence_df.columns if col != "coverage"]

In [63]:
plot_upset_coverage(presence_absence_df, ind_cols, run_color, save_path=f"{FIGURES_DIR}/supp_fig3e_{RUN_NAME}_upset_coverage.svg")

/opt/homebrew/Caskroom/miniconda/base/envs/instanexus/lib/python3.11/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  styles["linewidth"].fillna(1, inplace=True)
/opt/homebrew/Caskroom/miniconda/base/envs/instanexus/lib/python3.11/site-packages/upsetplot/plotting.py:796: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the inte

Plot saved: figures/supp_fig3e_bsa_upset_coverage.svg
